# Final Notebook - Classifying Spam Emails

Notebook cuối này tổng hợp nội dung từ các notebook `00` đến `06` và phần giao diện kiểm tra spam đã thêm vào project.

Mục tiêu của notebook:

- Tóm tắt bài toán phân loại email spam/not spam.
- Trình bày nguồn dữ liệu, chất lượng dữ liệu và cân bằng nhãn.
- Mô tả preprocessing, feature engineering và các model đã train.
- Hiển thị kết quả đánh giá, learning curve và confusion matrix.
- Demo dự đoán email mới bằng model đã train.
- Ghi lại cách chạy giao diện web local.


## 1. Problem Definition

Bài toán: xây dựng hệ thống phân loại email thành 2 lớp:

- `0`: not spam / ham
- `1`: spam

Input là nội dung email dạng text. Output là nhãn dự đoán `spam` hoặc `not spam`, kèm `spam_score` và các từ/cụm từ ảnh hưởng tới kết quả.

Với bài toán spam detection, nhóm không chỉ nhìn accuracy. Các chỉ số quan trọng gồm:

- `precision_spam`: khi model báo spam thì đúng bao nhiêu.
- `recall_spam`: model bắt được bao nhiêu email spam thật.
- `f1_spam`: cân bằng giữa precision và recall.
- `false_positive_rate`: tỷ lệ chặn nhầm email tốt.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'

print('Project root:', PROJECT_ROOT)
print('Reports exists:', REPORTS_DIR.exists())
print('Models exists:', MODELS_DIR.exists())

## 2. Data Collection

Nguồn dữ liệu chính của project:

| Nguồn | Vai trò |
| --- | --- |
| SpamAssassin Public Corpus | Nguồn ham/spam public, có nhãn rõ |
| SetFit Enron Spam | Dataset email spam/ham từ Hugging Face |
| locuoco 300k spam/ham/phish | Dataset lớn, map spam/phish về class spam |
| TREC Spam Track | Nguồn chính thống để tham khảo thêm |
| CMU Enron Email Dataset | Nguồn email Enron chính thống, cần gán nhãn cẩn thận nếu mở rộng |

Project chuẩn hóa dữ liệu về schema:

```text
source,file_name,label,label_name,subject,text
```


In [ ]:
source_path = PROJECT_ROOT / 'data_sources' / 'data_links.csv'
if source_path.exists():
    sources = pd.read_csv(source_path)
    display(sources)
else:
    print('Không tìm thấy data_sources/data_links.csv')


## 3. Data Quality và cân bằng dữ liệu

Sau khi gộp dữ liệu, project kiểm tra:

- Label không hợp lệ.
- Text rỗng.
- Text quá ngắn.
- Dòng trùng lặp theo `label` + `text`.

Sau khi lọc lỗi, dataset được cân bằng lại để số lượng spam và not spam ngang nhau. Việc cân bằng giúp model không thiên lệch về class có số lượng lớn hơn.


In [ ]:
quality_path = REPORTS_DIR / 'data_quality_report.json'
loader_path = REPORTS_DIR / 'data_loader_report.json'

for path in [loader_path, quality_path]:
    print('\n---', path.name, '---')
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        print(json.dumps(payload, indent=2, ensure_ascii=False))
    else:
        print('Không tìm thấy file report:', path)


## 4. Preprocessing

Module chính: `src/text_preprocess.py`

Các bước xử lý text:

- Xóa HTML, script, style.
- Chuẩn hóa URL thành `urltoken`.
- Chuẩn hóa email address thành `emailtoken`.
- Chuẩn hóa số thành `numbertoken`.
- Lowercase.
- Loại ký tự đặc biệt.
- Loại stopwords tiếng Anh nếu có NLTK corpus, dùng fallback nếu thiếu.


In [ ]:
from src.text_preprocess import clean_email_text

examples = [
    '<html>FREE prize!!! Click https://spam.example now, contact test@example.com for 100 dollars</html>',
    'Hi team, please confirm tomorrow meeting agenda and send the project report when ready.',
]

for text in examples:
    print('RAW:', text)
    print('CLEAN:', clean_email_text(text))
    print()

## 5. Feature Engineering và Model Training

Model được train bằng pipeline:

```text
TfidfVectorizer(max_features=25000, ngram_range=(1, 2), stop_words='english')
-> classifier
```

Ba model so sánh:

- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM

Model tốt nhất được lưu tại `models/spam_classifier.joblib`.


In [ ]:
metrics_path = REPORTS_DIR / 'model_metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)

    metric_cols = ['accuracy', 'precision_spam', 'recall_spam', 'f1_spam']
    ax = metrics.set_index('model')[metric_cols].plot(kind='bar', figsize=(10, 5), ylim=(0.9, 1.0))
    ax.set_title('So sánh metric giữa các model')
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('Không tìm thấy reports/model_metrics.csv')


## 6. Confusion Matrix

Các ảnh dưới đây là kết quả đánh giá model đã train, được sinh từ `src/model_evaluate.py` và lưu trong `reports/figures/`.


![Linear SVM confusion matrix](../reports/figures/linear_svm_confusion_matrix.png)

![Logistic Regression confusion matrix](../reports/figures/logistic_regression_confusion_matrix.png)

![Naive Bayes confusion matrix](../reports/figures/naive_bayes_confusion_matrix.png)

## 7. Learning Curve

Các ảnh learning curve dưới đây là kết quả đã train và đã được lấy lại từ quá trình đánh giá model. Đây không phải ảnh mẫu.

- Đường đỏ: F1-score trên tập train.
- Đường xanh: F1-score cross-validation.
- Khoảng cách giữa hai đường càng nhỏ thì model càng ít overfitting.


![Learning Curve - Linear SVM](../reports/figures/learning_curve_linear_svm.png)

![Learning Curve - Logistic Regression](../reports/figures/learning_curve_logistic_regression.png)

![Learning Curve - Naive Bayes](../reports/figures/learning_curve_naive_bayes.png)

## 8. Demo Predict Email mới

Module chính: `src/predict.py`

Hàm `predict_email()` trả về:

- `prediction`: `spam` hoặc `not spam`
- `spam_score`: điểm nghiêng về spam trong khoảng 0-1
- `cleaned_text`: text sau tiền xử lý


In [ ]:
from src.predict import load_pipeline, predict_email

model_path = MODELS_DIR / 'spam_classifier.joblib'
model = load_pipeline(model_path)

demo_emails = [
    'Congratulations winner, claim your free lottery prize money now by clicking this urgent link.',
    'Hi team, please confirm tomorrow meeting agenda and send the project report when ready.',
]

for email_text in demo_emails:
    result = predict_email(email_text, model=model)
    print('Email:', email_text)
    print('Prediction:', result['prediction'])
    print('Spam score:', result['spam_score'])
    print('Cleaned:', result['cleaned_text'])
    print()

## 9. Web UI kiểm tra spam

Project có giao diện Flask tại `app.py`.

Cách chạy:

```powershell
python app.py
```

Sau đó mở:

```text
http://127.0.0.1:5000
```

Lưu ý: không mở trực tiếp file `templates/spam_checker.html` bằng Live Server port `5500`, vì Flask cần render cú pháp Jinja như `{% ... %}` và `{{ ... }}`.


In [ ]:
sample_dir = PROJECT_ROOT / 'sample_emails'
if sample_dir.exists():
    for path in sorted(sample_dir.glob('*.txt')):
        print(path.name)
else:
    print('Không tìm thấy sample_emails/')


## 10. Kết luận

Pipeline cuối cùng đã có đủ các phần:

- Nguồn dữ liệu và kiểm tra nguồn.
- Data loading, quality check và cân bằng nhãn.
- Text preprocessing.
- Feature engineering bằng TF-IDF.
- Train và so sánh Naive Bayes, Logistic Regression, Linear SVM.
- Đánh giá bằng accuracy, precision, recall, F1-score và confusion matrix.
- Dự đoán email mới bằng model đã train.
- Giao diện web local để kiểm tra email spam/not spam.

Khi thuyết trình, nhóm nên nhấn mạnh rằng kết quả dự đoán dựa trên đặc trưng văn bản đã học từ dataset, không phải rule thủ công.
